<span style="color:pink; font-size:15px;">File to run tests for the simulation code:</span>
<span style="color:yellow; font-size:15px;">Used to verify that the functions in `lin_alg.py` and `unitary.py` are working correctly, and test the overall simulation workflow.</span>

In [60]:
import numpy as np
import matplotlib.pyplot as plt
from state_gen import generate_state
from energy import energy, passive_energy_g, passive_energy_l, ergotropy_gap, global_ergo, local_ergo
from local_to import bath_gibbs_states, LTO_step, LTO_step2
from unitary import generate_unitary
from lin_alg import negativity, partial_trace, vn_entropy, rel_entropy, passive_state, partial_transpose

In [61]:
w1=0.001
w2=0.001
# w3=1.5 #1.0 #1.5
# w4=0.5 #2.0 #0.5
w3 = 0.001
w4 = 0.001
beta_a=1.0
beta_b=2.0
a = 0.8
p = 0.5
n_terms = 4
N = np.sqrt(a * (1 - a))
np.set_printoptions(precision=4, suppress=True)

kind_dict = {
    1: "product",
    2: "separable",
    3: "schmidt_ent",
    4: "pure_ent",
    5: "werner",
    6: "mixed_ent",
    7: "random"
}

In [62]:
rho12 = generate_state(kind=kind_dict[4], a=a, n_terms=n_terms, p=p)
# gamma_a, gamma_b = bath_gibbs_states(beta_a, beta_b, w3, w4)
# rho12 = np.kron(gamma_a, gamma_b)
# set1 = np.random.rand(2)
# set2 = np.random.rand(2)
# print(f"set1: {set1/sum(set1)} and set2: {set2/sum(set2)}")
# rho12 = np.kron(np.diag((set1)/sum(set1)), np.diag((set2)/sum(set2)))
# print(f"rho12: \n{rho12}")
neg_rho12 = negativity(rho12)
energy_rho12 = energy(rho12, w1, w2)
global_pass_energy_rho12 = passive_energy_g(rho12, w1, w2)
local_pass_energy_rho12 = passive_energy_l(rho12, w1, w2) 
print(f"rho12: \n{rho12}")
print(f"Negativity: {neg_rho12}")
print(f"Energy: {energy_rho12}")
print(f"Global Passive Energy: {global_pass_energy_rho12}")
print(f"Local Passive Energy: {local_pass_energy_rho12}")
print(f"Ergotropy Gap: {global_pass_energy_rho12 - local_pass_energy_rho12}")

rho12: 
[[ 0.0822+0.j      0.1408-0.1942j  0.0603+0.0271j  0.0239-0.1139j]
 [ 0.1408+0.1942j  0.6999-0.j      0.0392+0.189j   0.3099-0.1387j]
 [ 0.0603-0.0271j  0.0392-0.189j   0.0532+0.j     -0.0201-0.0914j]
 [ 0.0239+0.1139j  0.3099+0.1387j -0.0201+0.0914j  0.1647-0.j    ]]
Negativity: 0.1446540143151081
Energy: 0.000165048737
Global Passive Energy: -0.002
Local Passive Energy: -0.00191447211
Ergotropy Gap: -8.552789000000009e-05


In [63]:
# rho_a1,rho_b1 = bath_gibbs_states(beta_a=1.0, beta_b=2.0, w3=1.0, w4=2.0)
# rho12 =  np.kron(rho_a1, rho_b1)
# print(rho12)

In [64]:
# unitary_type = "swap"
unitary_type = "deg1"
Ua = generate_unitary(unitary_type)
Ub = generate_unitary(unitary_type)
print(f"Ua: \n{Ua}")
print(f"Ub: \n{Ub}")

gamma_a, gamma_b = bath_gibbs_states(beta_a, beta_b, w3, w4)
# gamma_a = np.diag([0,1])
# gamma_b = np.diag([0,1])
print(f"gamma_a: \n{gamma_a}")
print(f"gamma_b: \n{gamma_b}")

# rho12 = np.kron(np.diag([1,0]), np.diag([1,0]))

Ua: 
[[-0.9606+0.2778j  0.    +0.j      0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j     -0.5517-0.3522j  0.2071+0.7271j  0.    +0.j    ]
 [ 0.    +0.j     -0.6037-0.4551j -0.1298-0.6415j  0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.    +0.j     -0.8325-0.554j ]]
Ub: 
[[-0.572 -0.8202j  0.    +0.j      0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j     -0.4507-0.8701j -0.1965-0.0344j  0.    +0.j    ]
 [ 0.    +0.j     -0.1495+0.132j  -0.0709-0.9773j  0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.    +0.j      0.6089-0.7933j]]
gamma_a: 
[[0.5005 0.    ]
 [0.     0.4995]]
gamma_b: 
[[0.501 0.   ]
 [0.    0.499]]


In [65]:
sz = np.array([[-1, 0], [0, 1]], dtype=complex)
I = np.eye(2, dtype=complex)

Htot1 = w1*np.kron(sz,I) + w3*np.kron(I,sz)
Htot2 = w2*np.kron(sz,I) + w4*np.kron(I,sz)

print(np.linalg.norm(Ua @ Htot1 - Htot1 @ Ua))
print(np.linalg.norm(Ub @ Htot2 - Htot2 @ Ub))
print(np.linalg.norm(Ub @ Htot1 - Htot1 @ Ub))

# # Build block unitary in eigenbasis
# def unitary_gen(Htot, tol=1e-10):

#     # Diagonalize Hamiltonian
#     E, V = np.linalg.eigh(Htot)

#     # eigh already sorts eigenvalues ascending,
#     # so this ensures "lowest energy first"
#     E = np.real(E)
#     dim = len(E)

#     # Prepare unitary in eigenbasis
#     Ueig = np.zeros((dim, dim), dtype=complex)

#     # Identify degenerate sectors
#     sectors = []
#     current = [0]

#     for i in range(1, dim):
#         if abs(E[i] - E[i-1]) < tol:
#             current.append(i)
#         else:
#             sectors.append(current)
#             current = [i]
#     sectors.append(current)

#     # Fill blocks
#     for inds in sectors:
#         d = len(inds)

#         if d == 1:
#             # single phase
#             Ueig[inds[0], inds[0]] = np.exp(1j * 2*np.pi*np.random.rand())
#         else:
#             # random unitary block
#             X = np.random.randn(d, d) + 1j*np.random.randn(d, d)
#             Q, _ = np.linalg.qr(X)
#             Ueig[np.ix_(inds, inds)] = Q

#     # Transform back to computational basis
#     U = V @ Ueig @ V.conj().T

#     return U

# Ua = unitary_gen(Htot1)
# Ub = unitary_gen(Htot2)
print(np.linalg.norm(Ua @ Htot1 - Htot1 @ Ua))
print(np.linalg.norm(Ub @ Htot2 - Htot2 @ Ub))
print(np.linalg.norm(Ub @ Htot1 - Htot1 @ Ub))
print(f"Ua: \n{Ua}")
print(f"Ub: \n{Ub}")
print(f"Ua @ Ua^dagger: \n{Ua @ Ua.conj().T}")
print(f"Ub @ Ub^dagger: \n{Ub @ Ub.conj().T}")

0.0
0.0
0.0
0.0
0.0
0.0
Ua: 
[[-0.9606+0.2778j  0.    +0.j      0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j     -0.5517-0.3522j  0.2071+0.7271j  0.    +0.j    ]
 [ 0.    +0.j     -0.6037-0.4551j -0.1298-0.6415j  0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.    +0.j     -0.8325-0.554j ]]
Ub: 
[[-0.572 -0.8202j  0.    +0.j      0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j     -0.4507-0.8701j -0.1965-0.0344j  0.    +0.j    ]
 [ 0.    +0.j     -0.1495+0.132j  -0.0709-0.9773j  0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.    +0.j      0.6089-0.7933j]]
Ua @ Ua^dagger: 
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j -0.+0.j  0.+0.j]
 [ 0.+0.j -0.-0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]
Ub @ Ub^dagger: 
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j -0.-0.j  0.+0.j]
 [ 0.+0.j -0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]


In [66]:
# check unitarity
print(f"Ua @ Ua^dagger: \n{Ua @ Ua.conj().T}")

Ua @ Ua^dagger: 
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j -0.+0.j  0.+0.j]
 [ 0.+0.j -0.-0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]


In [67]:
sigma, gamma_a1, gamma_b1 = LTO_step(rho12, gamma_a, gamma_b, Ua, Ub)
# sigma2 = LTO_step2(rho12, gamma_a, gamma_b, Ua, Ub)
print(f"rho12: \n{rho12}")
print(f"sigma: \n{sigma}\n")
# print(f"sigma2: \n{sigma2}\n")

print(f"Negativity of rho12: {negativity(rho12)}")
print(f"Negativity of sigma: {negativity(sigma)}\n")

print(f"Energy of rho12: {energy(rho12, w1, w2)}")
print(f"Energy of sigma: {energy(sigma, w1, w2)}\n")

print(f"Global Passive Energy of rho12: {passive_energy_g(rho12, w1, w2)}")
print(f"Global Passive Energy of sigma: {passive_energy_g(sigma, w1, w2)}\n")

print(f"Local Passive Energy of rho12: {passive_energy_l(rho12, w1, w2)}")
print(f"Local Passive Energy of sigma: {passive_energy_l(sigma, w1, w2)}\n")

print(f"Ergotropy Gap of rho12: {ergotropy_gap(rho12, w1, w2)}")
print(f"Ergotropy Gap of sigma: {ergotropy_gap(sigma, w1, w2)}\n")

print(f"Global Ergotropy gap after LTO: {global_ergo(sigma, w1, w2) - global_ergo(rho12, w1, w2)}")
print(f"Local Ergotropy gap after LTO: {local_ergo(sigma, w1, w2) - local_ergo(rho12, w1, w2)}\n")

rho12: 
[[ 0.0822+0.j      0.1408-0.1942j  0.0603+0.0271j  0.0239-0.1139j]
 [ 0.1408+0.1942j  0.6999-0.j      0.0392+0.189j   0.3099-0.1387j]
 [ 0.0603-0.0271j  0.0392-0.189j   0.0532+0.j     -0.0201-0.0914j]
 [ 0.0239+0.1139j  0.3099+0.1387j -0.0201+0.0914j  0.1647-0.j    ]]
sigma: 
[[ 0.0834+0.j     -0.0547-0.1696j  0.0275-0.0146j -0.0486-0.0052j]
 [-0.0547+0.1696j  0.5377+0.j      0.0167+0.0793j  0.0465-0.1421j]
 [ 0.0275+0.0146j  0.0167-0.0793j  0.0666+0.j     -0.0675-0.0939j]
 [-0.0486+0.0052j  0.0465+0.1421j -0.0675+0.0939j  0.3123+0.j    ]]

Negativity of rho12: 0.1446540143151081
Negativity of sigma: 0.0

Energy of rho12: 0.000165048737
Energy of sigma: 0.000457825459

Global Passive Energy of rho12: -0.002
Global Passive Energy of sigma: -0.001354242365

Local Passive Energy of rho12: -0.00191447211
Local Passive Energy of sigma: -0.001332540788

Ergotropy Gap of rho12: 8.552789000000009e-05
Ergotropy Gap of sigma: 2.1701576999999918e-05

Global Ergotropy gap after LTO: -0.000

In [68]:
rho_a = partial_trace(rho12, sys=1)
rho_b = partial_trace(rho12, sys=0)
pass_rho_a = passive_state(rho_a)
pass_rho_b = passive_state(rho_b)
gamma_a, gamma_b = bath_gibbs_states(beta_a, beta_b, w3, w4)
bound_l = float(np.round(rel_entropy(pass_rho_a, gamma_a)/beta_a + rel_entropy(pass_rho_b, gamma_b)/beta_b, 12))
print(f"Local bound: {bound_l}")

Local bound: 0.882753031978


In [69]:
print(f"Partial trace of rho12 (sys=1): \n{partial_trace(rho12, sys=1)}")
print(f"Partial trace of rho12 (sys=0): \n{partial_trace(rho12, sys=0)}")
print(f"Partial trace of sigma (sys=1): \n{partial_trace(sigma, sys=1)}")
print(f"Partial trace of sigma (sys=0): \n{partial_trace(sigma, sys=0)}")

Partial trace of rho12 (sys=1): 
[[0.7821+0.j     0.3702-0.1116j]
 [0.3702+0.1116j 0.2179-0.j    ]]
Partial trace of rho12 (sys=0): 
[[0.1354+0.j     0.1207-0.2856j]
 [0.1207+0.2856j 0.8646-0.j    ]]
Partial trace of sigma (sys=1): 
[[0.6211+0.j     0.074 -0.1567j]
 [0.074 +0.1567j 0.3789+0.j    ]]
Partial trace of sigma (sys=0): 
[[ 0.15  +0.j     -0.1222-0.2635j]
 [-0.1222+0.2635j  0.85  +0.j    ]]


In [70]:
sigma1 = partial_trace(sigma, sys=1)
sigma2 = partial_trace(sigma, sys=0)

print(np.allclose(sigma, np.kron(sigma1, sigma2), atol=1e-10))

False


In [71]:
rho1 = partial_trace(rho12, sys=1)
rho2 = partial_trace(rho12, sys=0)

print(np.allclose(rho12, np.kron(rho1, rho2)))


False


In [72]:
def entropy(rho):
    vals = np.linalg.eigvalsh(rho)
    vals = vals[vals > 1e-12]
    return -np.sum(vals * np.log(vals))

def free_energy(rho, H, beta):
    E = np.trace(H @ rho).real
    S = entropy(rho)
    return E - (S / beta)

rho_a = partial_trace(rho12, sys=1)
rho_b = partial_trace(rho12, sys=0)

sigma_a = partial_trace(sigma, sys=1)
sigma_b = partial_trace(sigma, sys=0)

Ha = w1*sz
Hb = w2*sz

Fa_before = free_energy(rho_a, Ha, beta_a)
Fa_after  = free_energy(sigma_a, Ha, beta_a)
    
Fb_before = free_energy(rho_b, Hb, beta_b)
Fb_after  = free_energy(sigma_b, Hb, beta_b)

print("Fa:", Fa_before, "->", Fa_after)
print("Fb:", Fb_before, "->", Fb_after)

print("Rel entropy with gamma_a before:", rel_entropy(pass_rho_a, gamma_a), "->", rel_entropy(sigma_a, gamma_a))
print("Rel entropy with gamma_b before:", rel_entropy(pass_rho_b, gamma_b), "->", rel_entropy(sigma_b, gamma_b))

def energy2(rho, w):
    return -w*rho[0,0].real + w*rho[1,1].real

print("Total energy of subsystem a before LTO:", energy2(rho_a, w1)+ energy2(gamma_a, w3))
print("Total energy of subsystem b before LTO:", energy2(rho_b, w2) + energy2(gamma_b, w4))
print("Total energy of subsystem a after LTO:", energy2(sigma_a, w1) + energy2(gamma_a1, w3))
print("Total energy of subsystem b after LTO:", energy2(sigma_b, w2) + energy2(gamma_b1, w4))


Fa: -0.10393398226634905 -> -0.6011278722518418
Fb: -0.05095573574835263 -> -0.0912715096339875
Rel entropy with gamma_a before: 0.588820600004168 -> 0.09201980830802027
Rel entropy with gamma_b before: 0.5878648639481383 -> 0.5106061612906371
Total energy of subsystem a before LTO: -0.0005651377651012564
Total energy of subsystem b before LTO: 0.0007271865047712952
Total energy of subsystem a after LTO: -0.0005651377651012563
Total energy of subsystem b after LTO: 0.0007271865047712951
